# Loyalty Program Dataset Generation Pipeline

In [ ]:
# # uncomment if using Google Colab
# # mount drive to access drive folders
# from google.colab import drive
# drive.mount('/content/drive')

## Load Dependencies

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, date
import matplotlib.pyplot as plt
import random

In [ ]:
# set seed for reproducibility
SEED = 21
np.random.seed(SEED)
random.seed(SEED)
rng = np.random.default_rng(SEED)
fake = Faker(['en-PH'])
fake.seed_instance(SEED)

# set global parameters

START_DATE = pd.to_datetime("2022-01-01")
END_DATE = datetime.today()

## Generate Members table with default Bronze tier assignment

In [ ]:
def generate_member_data(start_date, initial_members, growth_rate, seed, end_date = None):
    end_date = end_date if end_date is not None else datetime.now().date()
    rng = np.random.default_rng(seed)
    months = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
    # Calculate the number of members to generate
    N_MEMBERS = int(initial_members * (1+ growth_rate) ** months)

    # BIRTHDAY DISTRIBUTION
    young_ages = rng.normal(loc=28, scale=5, size=int(N_MEMBERS * 0.7)) # 70% of members are between 18-35, with a mean age of 28 and a std dev of 5
    older_ages = rng.normal(loc=45, scale=8, size= N_MEMBERS - int(N_MEMBERS * 0.7)) # 30% of members are between 36-65, with a mean age of 45 and a std dev of 8
    ages = np.concatenate([young_ages, older_ages])

    ages = np.clip(ages, 16, 65).astype(int)                            # limit ages to 16-65 and convert to integers
    rng.shuffle(ages)

    birth_years = [end_date.year - age for age in ages]           # calculate birth years based on ages

    dobs = [
        date(year, rng.integers(1, 13), rng.integers(1, 29))  # use rng for full reproducibility; day capped at 28 to avoid invalid dates
        for year in birth_years
    ]

    # JOIN DATE
    date_range = pd.date_range(start_date, end_date)
    t = np.linspace(0, 2*np.pi, len(date_range)) # create a time variable for the sine wave
    weights = 1.5 + np.sin(t)  # oscillates from 0.5 to 2.5
    weights = weights / weights.sum()

    join_dates = pd.to_datetime(
        rng.choice(date_range, size=N_MEMBERS, p=weights)
    )
    # GENDER
    genders = rng.choice(
        ["Male", "Female", "Other"],
        size=N_MEMBERS,
        p=[0.30, 0.50, 0.20]
    )

    first_names = []
    last_names = []

    # name based on gender
    for g in genders:
        if g == "Male":
            first_names.append(fake.first_name_male())
        elif g == "Female":
            first_names.append(fake.first_name_female())
        else:
            first_names.append(fake.first_name_nonbinary())

        last_names.append(fake.last_name())


    # EMAIL GENERATION
    email_domains = ["gmail.com","yahoo.com","outlook.com","icloud.com"]
    emails = []
    # generate emails based on names
    for i in range(N_MEMBERS):
        first = first_names[i].lower()
        last = last_names[i].lower()

        domain = rng.choice(email_domains)
        style = rng.choice([
            f"{first}.{last}",
            f"{first}{last}",
            f"{first}_{last}",
            f"{first}{last}{rng.integers(1, 100)}"
        ])

        email = f"{style}@{domain}"
        emails.append(email)

    members = pd.DataFrame({
        "member_id": range(1, N_MEMBERS + 1),
        "first_name": first_names,
        "last_name": last_names,
        "gender": genders,
        "email": emails,
        "dob": pd.to_datetime(dobs),
        "age": ages,
        "join_date": join_dates,
        "current_tier": None,
    })

    members["age"] = (
        (end_date - members["dob"]).dt.days // 365              # update age again to align with dob
    )

    members["tenure_months"] = (
        (end_date - members["join_date"]).dt.days / 30          # calculate tenure
    ).astype(int)

    members["current_tier"] = members["current_tier"].fillna("Bronze")  # assign Bronze tier default
    members.reset_index(drop=True, inplace=True)
    return members

In [ ]:
members = generate_member_data(START_DATE,
                               initial_members=100,
                               growth_rate=0.05,
                               seed=SEED,
                               end_date=END_DATE)
members.head(5)

## Simulate Transactions

In [ ]:
def generate_transaction_data(members, seed, end_date = None):
    rng = np.random.default_rng(seed)
    end_date = pd.Timestamp(end_date) if end_date is not None else pd.Timestamp(datetime.today())
    transaction_ids = []
    member_ids = []
    transaction_dates = []
    amounts = []
    item_counts = []
    discount_amounts = []

    transaction_id = 1
    members = members.sort_values("member_id").reset_index(drop=True).copy()
    for _, member in members.iterrows():
        tenure_months = member['tenure_months']
        # frequency distribution based on age and tenure
        if member["age"] < 30:                                          # younger members tend to have more transactions
            base_lambda = 20
        elif member["age"] < 45:                                        # middle-aged members have moderate transactions
            base_lambda = 15
        else:                                                           # older members have fewer transactions
            base_lambda = 5

        tenure_exponent = rng.uniform(0.5, 1.2)
        lambda_transactions = base_lambda * (tenure_months / 12)** tenure_exponent       # random tenure impact (for noise)
        n_member_tx = rng.poisson(max(lambda_transactions, 1))          # poisson distribution to determine number of transactions, w at least 1 expected transaction

        # loop over no. of transactions for each member
        for _ in range(n_member_tx):
            days_between = (end_date - member["join_date"]).days
            random_days = rng.uniform(0, days_between)
            tx_date = member['join_date'] + pd.Timedelta(days=random_days)
            # add time between 10:00 and 21:59 (10 AM to before 10 PM)
            hour = rng.integers(10, 22)      # 10–21
            minute = rng.integers(0, 60)
            second = rng.integers(0, 60)

            transaction_date = (tx_date + pd.Timedelta(
                hours=hour,
                minutes=minute,
                seconds=second
            )).round('s')  # round to nearest second for cleaner timestamps

            # apply weekly seasonality (weekends busier, weekdays slower)
            day_of_week = transaction_date.dayofweek
            if day_of_week >=4:  # Weekend
                keep_probability = rng.uniform(0.6, 1.0)  # 60-100% chance to keep weekend transactions
            else:
                keep_probability = rng.uniform(0.2, 0.5)  # 20-50% chance to keep weekday transactions

            # apply yearly seasonality (holidays and seasonal events)
            month = transaction_date.month
            if month in [11, 12]:  # November-December (Black Friday, Christmas)
                seasonal_multiplier = rng.uniform(1.5, 2.5)
            elif month in [2]:  # February (Valentine's)
                seasonal_multiplier = rng.uniform(1.2, 1.8)
            elif month in [9, 10]:  # September-October (Back-to-School)
                seasonal_multiplier = rng.uniform(1.1, 1.6)
            elif month in [6, 7]:  # June-July (Summer)
                seasonal_multiplier = rng.uniform(0.9, 1.3)
            elif month in [1, 8]:  # January, August (Post-holiday slump)
                seasonal_multiplier = rng.uniform(0.5, 0.8)
            else:  # Other months
                seasonal_multiplier = rng.uniform(0.8, 1.2)

            keep_probability = min(keep_probability * seasonal_multiplier, 1.0)  # adjust keep probability by seasonality, capped at 1.0

            if rng.random() > keep_probability:
                continue  # Skip this transaction based on day-of-week and seasonal pattern

            item_count = rng.poisson(2) + 1                             # expected no. of items per transaction (2), min 1 purchase
            item_price = rng.lognormal(mean=4.8, sigma =0.3)            # prices between 120 - 200 usually
            item_price = round(item_price / 5) * 5                      # round to nearest 5 for more realistic pricing
            amount = item_count * item_price
            amount = round(amount * 2) / 2                              # compute amount and round to nearest 0.5 for more realistic totals, can be adjusted as needed

            # discount
            has_discount = rng.random() < 0.3                           # 30% chance of discount
            if has_discount:
                discount_rate = rng.beta(2,8)
                discount_amount = amount * discount_rate
            else:
                discount_amount = 0

            discount_amount = round(discount_amount * 2) / 2
            amount -= discount_amount

            transaction_ids.append(transaction_id)
            member_ids.append(member["member_id"])
            transaction_dates.append(transaction_date)
            amounts.append(round(amount, 2))
            item_counts.append(item_count)
            discount_amounts.append(discount_amount)

            transaction_id += 1

    transactions = pd.DataFrame({
        "transaction_id": transaction_ids,
        "member_id": member_ids,
        "transaction_date": transaction_dates,
        "amount": amounts,
        "item_count": item_counts,
        "discount_amount": discount_amounts
    })

    transactions.sort_values("transaction_date", inplace=True)
    transactions.reset_index(drop=True, inplace=True)
    transactions['transaction_id'] = transactions.index + 1
    return transactions

In [ ]:
transactions = generate_transaction_data(members, seed=SEED, end_date=END_DATE)
transactions.head(5)

## Assign Tiers

In [ ]:
def generate_tiers_assignment(members, transactions, end_date = None):
    end_date = (
        pd.Timestamp(end_date) if end_date is not None
        else pd.Timestamp(datetime.today())
    )
    cutoff_date = end_date


    # STEP 1: Define windows
    last_12m_start = cutoff_date - pd.DateOffset(months=12)  # F & M
    last_6m_start  = cutoff_date - pd.DateOffset(months=6)   # S
    last_3m_start  = cutoff_date - pd.DateOffset(months=3)   # Demotion


    # merge join date once for filtering in both 12m and 6m windows
    # more efficient than merging multiple times for each window separately
    tx = transactions.merge(
        members[["member_id", "join_date"]], on="member_id", how="left"
    )

    tx_12m = tx[
        (tx["transaction_date"] >= last_12m_start) &
        (tx["transaction_date"] <= cutoff_date)
    ]

    tx_6m = tx[
        (tx["transaction_date"] >= last_6m_start) &
        (tx["transaction_date"] <= cutoff_date)
    ].copy()
    # aggregate metrics for 12m window

    agg_12m = tx_12m.groupby("member_id").agg(
        F_12m=("transaction_id", "count"),
        M_12m=("amount",         "sum")
    ).reset_index()
    # aggregate metrics for 6m window (S = streak)
    tx_6m["year_month"] = tx_6m["transaction_date"].dt.to_period("M")
    active_months = (
        tx_6m.groupby(["member_id", "year_month"])
        .size()
        .reset_index()
        .groupby("member_id")
        .size()
        .reset_index(name="S_6m")
    )
    # STEP 4: Merge metrics into members
    df = members.copy() # copy for calculations
    df = df.merge(agg_12m,      on="member_id", how="left")
    df = df.merge(active_months, on="member_id", how="left")

    df["F_12m"] = df["F_12m"].fillna(0)
    df["M_12m"] = df["M_12m"].fillna(0)
    df["S_6m"]  = df["S_6m"].fillna(0)


    # STEP 5: Normalize metrics (0-1 scale)
    def minmax(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-6)

    df["F_norm"] = minmax(df["F_12m"])
    df["M_norm"] = minmax(df["M_12m"])
    df["S_norm"] = minmax(df["S_6m"])


    # STEP 6: Composite score
    df["tier_score"] = 0.5*df["F_norm"] + 0.4*df["M_norm"] + 0.1*df["S_norm"]


    # STEP 7: Assign tiers using FIXED POINT RANGES (~70/20/10 distribution) # changed this to percentiles
    gold_threshold   = df["tier_score"].quantile(0.90)
    silver_threshold = df["tier_score"].quantile(0.70)

    # changed to np.select for efficiency
    df["current_tier"] = np.select(
        [
            df["tier_score"] >= gold_threshold,
            df["tier_score"] >= silver_threshold,
        ],
        ["Gold", "Silver"],
        default="Bronze"
    )


    # STEP 8: Demotion for inactivity (last 3 months)
    recent_tx = transactions[
        (transactions["transaction_date"] >= last_3m_start) &
        (transactions["transaction_date"] <= cutoff_date)
    ]
    recent_activity = recent_tx.groupby("member_id").size().reset_index(name="recent_tx_count")
    df = df.merge(recent_activity, on="member_id", how="left")
    df["recent_tx_count"] = df  ["recent_tx_count"].fillna(0)
    # changed this to use np.where (does the same thing efficiently)
    df["current_tier"] = np.where(
        (df["recent_tx_count"] == 0) & (df["current_tier"] == "Gold"),
        "Silver",
        np.where(
            (df["recent_tx_count"] == 0) & (df["current_tier"] == "Silver"),
            "Bronze",
            df["current_tier"]
        )
    )
    # STEP 9: Final member-level tier table
    members = members.copy() # copy for assignment
    # assigned tier only, no need to write back all variables used sa computaioons
    members["current_tier"] = df["current_tier"].values

    # STEP 10: Tier summary table
    tier_counts = members.groupby("current_tier").size().reset_index(name="member_count")
    tiers = pd.DataFrame({
        "tier_id":   [1, 2, 3],
        "tier_name": ["Bronze", "Silver", "Gold"],
    })
    tiers = tiers.merge(
        tier_counts, left_on="tier_name", right_on="current_tier", how="left"
    )
    tiers["member_count"] = tiers["member_count"].fillna(0).astype(int)
    tiers.drop(columns=["current_tier"], inplace=True)
    tiers['point_multiplier'] = [1.0, 1.2, 1.5]
    return members, tiers

In [ ]:
# update members and generate tier table
members, tiers = generate_tiers_assignment(members, transactions, end_date=END_DATE)
members.head(5)

In [ ]:
tiers.head()

## Define Rewards

In [ ]:
def generate_rewards(start_date, end_date, seed=SEED):


    rng = np.random.default_rng(seed)
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    reward_pool = [
        ("Free Brewed Coffee", "Food", "Bronze", 150),
        ("Free Iced Coffee", "Food", "Bronze", 180),
        ("Free Pastry", "Food", "Bronze", 200),
        ("Free Drink Upsize", "Food", "Bronze", 120),
        ("Free Sticker Pack", "Merch", "Bronze", 100),
        ("Tote Bag", "Merch", "Bronze", 1500),
        ("Reusable Cup", "Merch", "Silver", 2000),
        ("Planner Set", "Merch", "Silver", 2000),
        ("Limited Edition Tumbler", "Merch", "Gold", 3000),
    ]

    rows = []
    reward_id = 1

    # number of rotation cycles
    cycle_start = start_date

    while cycle_start <= end_date:
        # each cycle lasts around 2–4 months
        cycle_days = int(rng.integers(60, 121))
        cycle_end = min(cycle_start + pd.Timedelta(days=cycle_days), end_date)

        # choose 4–6 rewards available in this cycle
        n_rewards = int(rng.integers(4, 7))
        selected = rng.choice(len(reward_pool), size=n_rewards, replace=False)

        for idx in selected:
            reward_name, reward_type, min_tier, base_cost = reward_pool[idx]

            cost_variation = rng.choice([-50, 0, 50, 100])
            points_cost = max(100, base_cost + cost_variation)

            rows.append({
                "reward_id": reward_id,
                "reward_name": reward_name,
                "reward_type": reward_type,
                "min_membership_tier": min_tier,
                "points_cost": int(points_cost),
                "valid_start": cycle_start,
                "valid_end": cycle_end,
            })

            reward_id += 1

        # next cycle starts after current cycle
        cycle_start = cycle_end + pd.Timedelta(days=1)

    # birthday reward
    rows.append({
        "reward_id": reward_id,
        "reward_name": "Birthday Free Pastry",
        "reward_type": "Promo",
        "min_membership_tier": "Bronze",
        "points_cost": 0,
        "valid_start": pd.NaT,
        "valid_end": pd.NaT,

    })

    rewards = pd.DataFrame(rows)

    birthday_reward = rewards[rewards["reward_name"] == "Birthday Free Pastry"].iloc[0]

    return rewards, birthday_reward

In [ ]:
rewards, birthday_reward = generate_rewards(START_DATE, END_DATE, seed=SEED)
rewards.head()

In [ ]:
birthday_reward

## Generate Points Ledger

In [ ]:
def generate_points_ledger(
    members,
    transactions,
    rewards,
    birthday_reward,
    tiers,
    start_date,
    end_date,
    expiry_days=365,
    burn_lambda=0.5,
    seed=SEED):
    rng = np.random.default_rng(seed)

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    TIER_ORDER = {"Bronze": 1, "Silver": 2, "Gold": 3}

    member_lookup = (
        members.set_index("member_id")[["current_tier", "join_date", "dob"]]
        .to_dict("index")
    )

    tier_multiplier = tiers.set_index("tier_name")["point_multiplier"].to_dict()

    ledger_data = []

    # EARN EVENTS

    for _, tx in transactions.iterrows():
        member_id = tx["member_id"]
        tier = member_lookup[member_id]["current_tier"]
        multiplier = tier_multiplier.get(tier, 1.0)

        points_earned = int(round(tx["amount"] * multiplier))

        ledger_data.append({
            "member_id": member_id,
            "transaction_id": tx["transaction_id"],
            "reward_id": None,
            "event_type": "EARN",
            "point_type": "TRANSACTION",
            "points_change": points_earned,
            "event_date": tx["transaction_date"]
        })

        # rolling expiry after 1 year
        expiry_date = tx["transaction_date"] + pd.Timedelta(days=expiry_days)

        if expiry_date <= end_date:
            ledger_data.append({
                "member_id": member_id,
                "transaction_id": tx["transaction_id"],
                "reward_id": None,
                "event_type": "EXPIRE",
                "point_type": "EXPIRY",
                "points_change": -points_earned,
                "event_date": expiry_date
            })

    points_ledger = pd.DataFrame(ledger_data)

    # REDEMPTION EVENTS

    burn_records = []

    for member_id in members["member_id"]:
        info = member_lookup[member_id]
        join_date = pd.to_datetime(info["join_date"])
        member_tier = info["current_tier"]
        member_rank = TIER_ORDER.get(member_tier, 1)

        member_events = points_ledger[points_ledger["member_id"] == member_id]
        balance = int(member_events["points_change"].sum())

        if balance <= 0:
            continue

        n_redemptions = rng.poisson(burn_lambda)

        for _ in range(n_redemptions):

            valid_rewards = rewards[
                (rewards["reward_id"] != birthday_reward["reward_id"]) &
                (rewards["points_cost"] <= balance) &
                (rewards["min_membership_tier"].map(TIER_ORDER) <= member_rank) &
                (rewards["valid_end"].notna()) &
                (rewards["valid_end"] >= join_date) &
                (rewards["valid_start"] <= end_date)
            ]

            if valid_rewards.empty:
                break

            reward = valid_rewards.sample(
                1,
                random_state=int(rng.integers(0, 99999))
            ).iloc[0]

            window_start = max(pd.to_datetime(reward["valid_start"]), join_date)
            window_end = min(pd.to_datetime(reward["valid_end"]), end_date)

            if window_start > window_end:
                continue

            window_days = max(1, (window_end - window_start).days + 1)
            burn_date = window_start + pd.Timedelta(
                days=int(rng.integers(0, window_days))
            )

            burn_records.append({
                "member_id": member_id,
                "transaction_id": None,
                "reward_id": int(reward["reward_id"]),
                "event_type": "BURN",
                "point_type": "REDEEM",
                "points_change": -int(reward["points_cost"]),
                "event_date": burn_date
            })

            balance -= int(reward["points_cost"])


    # BIRTHDAY REWARDS

    birthday_records = []

    for member_id, info in member_lookup.items():
        dob = pd.to_datetime(info["dob"])
        join_date = pd.to_datetime(info["join_date"])

        for year in range(start_date.year, end_date.year + 1):
            try:
                birthday_date = dob.replace(year=year)
            except ValueError:
                birthday_date = pd.Timestamp(f"{year}-02-28")

            if join_date <= birthday_date <= end_date:
                birthday_records.append({
                    "member_id": int(member_id),
                    "transaction_id": None,
                    "reward_id": int(birthday_reward["reward_id"]),
                    "event_type": "BURN",
                    "point_type": "BIRTHDAY_REWARD",
                    "points_change": 0,
                    "event_date": birthday_date
                })


    #  FINAL LEDGER
    extra_records = burn_records + birthday_records

    if extra_records:
        points_ledger = pd.concat(
            [points_ledger, pd.DataFrame(extra_records)],
            ignore_index=True
        )

    points_ledger = points_ledger.sort_values(
        ["member_id", "event_date"]
    ).reset_index(drop=True)

    points_ledger.insert(0, "ledger_id", range(1, len(points_ledger) + 1))

    points_ledger["running_balance"] = (
        points_ledger.groupby("member_id")["points_change"].cumsum()
    )

    return points_ledger

In [ ]:
points_ledger = generate_points_ledger(
    members,
    transactions,
    rewards,
    birthday_reward,
    tiers,
    start_date=START_DATE,
    end_date=END_DATE,
    expiry_days=365,
    burn_lambda=0.5,
    seed=SEED
)
points_ledger.head(5)

## Run All

In [ ]:
def generate_all_data():
    members = generate_member_data(START_DATE, initial_members=100, growth_rate=0.05, seed=SEED, end_date=END_DATE)
    transactions = generate_transaction_data(members, seed=SEED, end_date=END_DATE)
    members, tiers = generate_tiers_assignment(members, transactions, end_date=END_DATE)
    rewards, birthday_reward = generate_rewards(START_DATE, END_DATE, SEED)
    points_ledger = generate_points_ledger(
        members,
        transactions,
        rewards,
        birthday_reward,
        tiers,
        START_DATE,
        END_DATE,
        expiry_days=365,
        burn_lambda=0.5,
        seed=SEED
    )
    return members, transactions, tiers, rewards, points_ledger

In [ ]:
members, transactions, tiers, rewards, points_ledger = generate_all_data()

## Final Tables

In [ ]:
display(members.head(), tiers.head(), rewards.head(), transactions.head(), points_ledger.head())

## Export
Save tables as csv files

In [ ]:
import os
# # uncomment to save to Google Drive if using Colab
# # create folder in google drive for datasets
# folder = "/content/drive/MyDrive/Eskwelabs_Data"
# os.makedirs(folder, exist_ok=True)

# save as csv to created folder
# members.to_csv(f"{folder}/members.csv", index=False)
# transactions.to_csv(f"{folder}/transactions.csv", index=False)
# points_ledger.to_csv(f"{folder}/points_ledger.csv", index=False)
# rewards.to_csv(f"{folder}/rewards.csv", index=False)
# tiers.to_csv(f"{folder}/tiers.csv", index=False)

In [ ]:
# save files locally
path = "data/"
os.makedirs(path, exist_ok=True)
members.to_csv(f"{path}/members.csv", index=False)
transactions.to_csv(f"{path}/transactions.csv", index=False)
points_ledger.to_csv(f"{path}/points_ledger.csv", index=False)
rewards.to_csv(f"{path}/rewards.csv", index=False)
tiers.to_csv(f"{path}/tiers.csv", index=False)
